# 07 — Error Analysis

**Research**: Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF

**Objective**: Analyze misclassifications to understand which classes are difficult, which are frequently confused, and identify common patterns in failures.

**Models Analyzed**:
1. Multinomial Naive Bayes
2. Logistic Regression
3. Linear SVM

**Note**: This stage does *not* re-evaluate standard metrics (F1, Accuracy). It strictly analyzes the distribution and characteristics of the errors.

---

## 1. Setup

In [1]:
import sys
import os

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import project modules
from src.config.settings import ERROR_ANALYSIS_REPORTS_DIR
from src.config.error_analysis_config import ErrorAnalysisConfig
from src.error_analysis.error_loader import load_and_prepare_errors
from src.error_analysis.misclassification import analyze_misclassifications
from src.error_analysis.class_analysis import analyze_class_errors, identify_difficult_classes
from src.error_analysis.confusion_analysis import analyze_confusions, aggregate_top_confusions
from src.error_analysis.text_pattern_analysis import analyze_text_characteristics, find_frequent_error_words
from src.error_analysis.error_statistics import generate_error_statistics
from src.error_analysis.report_generator import generate_error_summary
from src.error_analysis.persistence import save_error_analysis_artifacts
import pandas as pd

print('Setup complete.')

Setup complete.


---
## 2. Load Predictions and Dataset

In [2]:
# This will load predictions if they exist in reports/evaluation/predictions/,
# or generate them automatically using the tuned models.
df_errors = load_and_prepare_errors()
display(df_errors.head(3))

Loading data for Error Analysis...
  Loading labels and text using exact split...
✓ Dataset split: Train=33,244, Test=8,312
  Split ratio: 80.0% / 20.0%
✓ Artifacts loaded: X_train=(33244, 22695), X_test=(8312, 22695), vocab=22,695
✓ Features validated: X_train=(33244, 22695), X_test=(8312, 22695), labels=6 classes
  Loading existing predictions for naive_bayes...
  Loading existing predictions for logistic_regression...
  Loading existing predictions for svm...
✓ Error Analysis dataset ready. (8312 samples)


,text,true_label,pred_naive_bayes,pred_logistic_regression,pred_svm
0,argumen netral jajah israel palestina ketidakt...,normal,normal,normal,normal
1,gubernur jatim target jembatan widang selesai ...,normal,normal,normal,normal
2,maze newsfeeds israel aku pindah paksa juta wa...,normal,normal,normal,normal


---
## 3. Analyze Misclassifications & Statistics

In [3]:
misclass_results = analyze_misclassifications(df_errors)

stats_df = generate_error_statistics(misclass_results)
print("\n--- Overall Error Statistics ---")
display(stats_df)


--- Overall Error Statistics ---


,total_samples,correct_predictions,errors,correct_percentage,error_percentage
model,,,,,
naive_bayes,8312,6564,1748,78.970164,21.029836
logistic_regression,8312,6556,1756,78.873917,21.126083
svm,8312,6735,1577,81.027430,18.972570


---
## 4. Class Difficulty Analysis

In [4]:
class_errors_dfs = {}
for m in stats_df.index:
    class_errors_dfs[m] = analyze_class_errors(df_errors, f"pred_{m}")

difficult_classes_df = identify_difficult_classes(class_errors_dfs)
print("\n--- Most Difficult Classes (Ranked by Recall Error Rate) ---")
display(difficult_classes_df)


--- Most Difficult Classes (Ranked by Recall Error Rate) ---


,naive_bayes_fn_rate,logistic_regression_fn_rate,svm_fn_rate,avg_recall_error_rate
class,,,,
threat,0.990698,0.939535,0.962791,0.964341
sexually_explicit,0.888889,0.833333,0.777778,0.833333
insult,0.713864,0.662242,0.710914,0.695674
harassment,0.611111,0.496032,0.456349,0.521164
hate_speech,0.397665,0.369505,0.370192,0.379121
normal,0.053048,0.075004,0.038644,0.055565


---
## 5. Confusion Pattern Analysis

In [5]:
confusion_dfs = {}
for m in stats_df.index:
    confusion_dfs[m] = analyze_confusions(df_errors, f"pred_{m}")

top_confusions_df = aggregate_top_confusions(confusion_dfs)
print("\n--- Top Confused Class Pairs (Aggregated) ---")
display(top_confusions_df.head(10))


--- Top Confused Class Pairs (Aggregated) ---


,true_label,predicted_label,count
3,hate_speech,normal,1447
6,insult,normal,1001
7,normal,hate_speech,675
10,threat,normal,502
5,insult,hate_speech,352
1,harassment,normal,261
8,normal,insult,206
2,hate_speech,insult,167
9,threat,hate_speech,79
0,harassment,hate_speech,61


---
## 6. Text Pattern Analysis

In [6]:
# We will analyze text patterns for the best performing model (assumed to be SVM or LR from previous steps).
# Let's just use the model with the lowest error percentage:
best_model_name = stats_df['error_percentage'].idxmin()
print(f"Analyzing text patterns for {best_model_name}...")

best_model_data = misclass_results[best_model_name]
text_stats = analyze_text_characteristics(best_model_data["correct_df"], best_model_data["misclassified_df"])
print("\nText Length & Token Count:")
print(text_stats)

print("\nFrequent Words in Errors:")
frequent_words_df = find_frequent_error_words(best_model_data["misclassified_df"], top_k=15)
display(frequent_words_df)

Analyzing text patterns for svm...

Text Length & Token Count:
{'correct': {'avg_char_length': np.float64(141.53956941351151), 'avg_token_count': np.float64(22.451967334818114)}, 'incorrect': {'avg_char_length': np.float64(109.0856055802156), 'avg_token_count': np.float64(17.72733037412809)}}

Frequent Words in Errors:


,word,error_frequency
0,israel,479
1,orang,311
2,palestina,195
3,indonesia,193
4,gaza,128
5,gila,123
6,dukung,122
7,negara,110
8,islam,109
9,anak,100


---
## 7. Generate Reports and Visualizations

In [7]:
print('Generating summary and saving artifacts...\n')

config = ErrorAnalysisConfig()

summary_md = generate_error_summary(stats_df, difficult_classes_df, top_confusions_df)

# Extract the raw error dataframes for saving
misclassified_dfs = {m: data["misclassified_df"] for m, data in misclass_results.items()}

save_error_analysis_artifacts(
    stats_df=stats_df,
    difficult_classes_df=difficult_classes_df,
    top_confusions_df=top_confusions_df,
    frequent_words_df=frequent_words_df,
    misclassified_dfs=misclassified_dfs,
    summary_md=summary_md,
    config=config
)

print(f'\n✓ Error Analysis complete. Results saved to {ERROR_ANALYSIS_REPORTS_DIR}')

Generating summary and saving artifacts...

  ✓ All error analysis artifacts saved to /home/zapp/Kampus/PM/reports/error_analysis

✓ Error Analysis complete. Results saved to /home/zapp/Kampus/PM/reports/error_analysis


---
## Summary

Error Analysis is complete.

**Generated Artifacts**:
- `reports/error_analysis/error_summary.md`
- Tables: `error_statistics.csv`, `difficult_classes.csv`, `confusion_analysis.csv`
- Visualizations: `figures/error_distribution.png`, `figures/top_confused_classes.png`
- Raw Errors: `raw/*_misclassified.csv`

**Next Step**: `08_explainability.ipynb` — Model Explainability (SHAP/LIME)